In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

In [32]:
df = pd.read_csv("Datos_regresion.csv").drop("Diag.Egr1", axis=1)

# Separar características y target
X = df.drop(["Fallece"], axis=1)
y = df["Fallece"]

In [33]:
categorical_cols = ["Diag.Ing1", "Diag.Ing2", "Diag.Egr2"]
numerical_cols = ["Edad", "APACHE", "TiempoVAM"]

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist"), categorical_cols),
        ("num", RobustScaler(), numerical_cols)
    ]
)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

In [36]:
from collections import Counter
from imblearn.over_sampling import SMOTE,ADASYN,BorderlineSMOTE, RandomOverSampler
print('Original dataset shape %s' % Counter(y))
sm = ADASYN(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 155})


In [37]:
print('Original dataset shape %s' % Counter(y))
sm = SMOTE(random_state=42)
X_smote, y_smote = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_smote))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 158})


In [38]:
print('Original dataset shape %s' % Counter(y))
sm = BorderlineSMOTE(random_state=42)
X_bs, y_bs = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_bs))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 158})


In [39]:
print('Original dataset shape %s' % Counter(y))
sm = RandomOverSampler(random_state=42)
X_ro, y_ro = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_ro))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 158})


In [40]:
pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("mlp", LogisticRegression())
])

In [41]:
pipeline.fit(X_res, y_res)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['Diag.Ing1', 'Diag.Ing2',
                                                   'Diag.Egr2']),
                                                 ('num', RobustScaler(),
                                                  ['Edad', 'APACHE',
                                                   'TiempoVAM'])])),
                ('mlp', LogisticRegression())])

In [42]:
y_pred = pipeline.predict(X_test)

In [43]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")

              precision    recall  f1-score   support

           0       1.00      0.72      0.84        18
           1       0.38      1.00      0.55         3

    accuracy                           0.76        21
   macro avg       0.69      0.86      0.69        21
weighted avg       0.91      0.76      0.80        21

AUC-ROC: 0.86


In [44]:
confusion_matrix(y_test, y_pred)

array([[13,  5],
       [ 0,  3]], dtype=int64)

# Comparando aumento de datos

In [45]:
pipeline.fit(X_smote, y_smote)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.67      0.80        18
           1       0.33      1.00      0.50         3

    accuracy                           0.71        21
   macro avg       0.67      0.83      0.65        21
weighted avg       0.90      0.71      0.76        21

AUC-ROC: 0.83
[[12  6]
 [ 0  3]]


In [46]:
pipeline.fit(X_bs, y_bs)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.67      0.80        18
           1       0.33      1.00      0.50         3

    accuracy                           0.71        21
   macro avg       0.67      0.83      0.65        21
weighted avg       0.90      0.71      0.76        21

AUC-ROC: 0.83
[[12  6]
 [ 0  3]]


In [47]:
pipeline.fit(X_ro, y_ro)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.78      0.85        18
           1       0.33      0.67      0.44         3

    accuracy                           0.76        21
   macro avg       0.63      0.72      0.65        21
weighted avg       0.85      0.76      0.79        21

AUC-ROC: 0.72
[[14  4]
 [ 1  2]]


# Prueba de campo

In [48]:
df = pd.read_csv("pc.csv", delimiter=";").drop(
    ["Pos.","Sexo","Piel","F.Ingreso UCI",
     "Fue un traslado","Procedencia","F.Egreso UCI",
     "Modos VAM"], axis=1)

# Separar características y target
X = df.drop("Fallece", axis=1)
X['TiempoVAM'] = X["TiempoVAM"]/24
y = df["Fallece"]

In [49]:
pipeline.predict_proba(pd.DataFrame(
    {
        'Edad':[60,69,78,47], 
        'Diag.Ing1':[16,8,14,3], 
        'Diag.Ing2':[21,14,16,0], 
        'Diag.Egr2':[17,0,12,0], 
        'APACHE':[16,16,23,9], 
        'TiempoVAM':[24,6,20,5]
    }
))

array([[0.55602786, 0.44397214],
       [0.44325441, 0.55674559],
       [0.05179807, 0.94820193],
       [0.67026947, 0.32973053]])

In [50]:
pipeline.predict_proba(X)
y_t = pipeline.predict(X)

In [51]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
confusion_matrix(y, y_t)

array([[50, 23],
       [ 1,  2]], dtype=int64)

In [52]:
print(classification_report(y, y_t))
print(f"AUC-ROC: {roc_auc_score(y, y_t):.2f}")

              precision    recall  f1-score   support

           0       0.98      0.68      0.81        73
           1       0.08      0.67      0.14         3

    accuracy                           0.68        76
   macro avg       0.53      0.68      0.47        76
weighted avg       0.94      0.68      0.78        76

AUC-ROC: 0.68


# Freezing

In [53]:
from joblib import dump

dump(pipeline, "new_workflow_lr.joblib")

['new_workflow_lr.joblib']

In [54]:
from joblib import load

model = load('new_workflow_lr.joblib')

In [55]:
model.predict_proba(pd.DataFrame(
    {
        'Edad':[60,69,78,47], 
        'Diag.Ing1':[16,8,14,3], 
        'Diag.Ing2':[21,14,16,0], 
        'Diag.Egr2':[17,0,12,0], 
        'APACHE':[16,16,23,9], 
        'TiempoVAM':[24,6,20,5]
    }
))

array([[0.55602786, 0.44397214],
       [0.44325441, 0.55674559],
       [0.05179807, 0.94820193],
       [0.67026947, 0.32973053]])

In [56]:
X_res

,Edad,Diag.Ing1,Diag.Ing2,Diag.Egr2,APACHE,TiempoVAM
0,56,14,0,0,14,2
1,73,14,16,16,24,8
2,57,1,0,0,4,2
3,65,18,0,0,17,2
4,42,3,7,7,19,5
...,...,...,...,...,...,...
308,74,11,0,2,17,7
309,79,14,16,16,23,3
310,77,14,16,16,22,3
311,85,14,17,16,21,2


In [57]:
X

,Edad,Diag.Ing1,Diag.Ing2,Diag.Egr2,APACHE,TiempoVAM
0,47,3,0,0,9,5.0
1,35,31,0,0,12,4.0
2,71,15,16,0,15,4.0
3,59,7,0,0,4,1.0
4,69,14,16,16,25,1.0
...,...,...,...,...,...,...
71,61,31,0,43,14,5.0
72,53,40,16,17,15,34.0
73,21,15,16,8,17,3.0
74,59,7,3,0,18,8.0


In [58]:
pipeline.predict_proba(X)[y!=y_t]

array([[0.13058859, 0.86941141],
       [0.33692462, 0.66307538],
       [0.09979443, 0.90020557],
       [0.19880524, 0.80119476],
       [0.34174088, 0.65825912],
       [0.4828712 , 0.5171288 ],
       [0.55602786, 0.44397214],
       [0.15219395, 0.84780605],
       [0.30509483, 0.69490517],
       [0.33392061, 0.66607939],
       [0.42763883, 0.57236117],
       [0.26949737, 0.73050263],
       [0.21740614, 0.78259386],
       [0.477109  , 0.522891  ],
       [0.22889727, 0.77110273],
       [0.33554375, 0.66445625],
       [0.39702366, 0.60297634],
       [0.31277225, 0.68722775],
       [0.34602687, 0.65397313],
       [0.49586053, 0.50413947],
       [0.40843932, 0.59156068],
       [0.23120475, 0.76879525],
       [0.3883063 , 0.6116937 ],
       [0.3295991 , 0.6704009 ]])

In [59]:
X[y!=y_t]

,Edad,Diag.Ing1,Diag.Ing2,Diag.Egr2,APACHE,TiempoVAM
4,69,14,16,16,25,1.000000
5,72,18,0,0,21,1.000000
6,70,14,12,0,23,3.000000
7,73,16,0,41,22,14.000000
10,54,3,7,0,19,3.000000
19,27,3,7,0,16,10.000000
22,60,16,21,17,16,24.000000
25,57,7,0,0,25,4.000000
31,81,14,16,0,17,10.000000
33,79,46,0,0,17,5.000000
